# Chapter 5 — NumPy and Index Numbers

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch05_numpy_index_numbers.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 5.1 From lists to arrays

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from indiadata import mospi_cpi

cpi = mospi_cpi(sector="combined")
cpi_2025 = cpi[cpi["date"].dt.year == 2025]["index"].tolist()
print(cpi_2025)

In [ ]:
print(cpi_2025[:3] * 2)                # a list: repeated
a = np.array(cpi_2025)
print(a[:3] * 2)                       # an array: doubled
print("dtype:", a.dtype, "| shape:", a.shape)

rebased = a / a[0] * 100
mom = (a[1:] / a[:-1] - 1) * 100       # each month over the previous month
print(np.round(rebased, 2))
print(np.round(mom, 2))

## 5.2 Why vectorise? A speed test

In [ ]:
import math
import time

g = np.linspace(0.001, 0.15, 1_000_000)     # 1,000,000 evenly spaced rates

t0 = time.perf_counter()
years_loop = [math.log(2) / math.log(1 + x) for x in g]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
years_vec = np.log(2) / np.log1p(g)         # log1p(g) = log(1 + g), accurately
t_vec = time.perf_counter() - t0

print(f"loop:       {t_loop:.3f} s")
print(f"vectorised: {t_vec:.3f} s   ({t_loop / t_vec:.0f} times faster)")
print("same answers:", np.allclose(years_loop, years_vec))
del years_loop                              # free the memory of the big list

## 5.3 Aggregation and boolean masks

In [ ]:
from indiadata import mospi_iip

iip_old = mospi_iip(base_year="2011-12")
gen_old = iip_old[iip_old["type"] == "General"].sort_values("date")
idx = gen_old["index"].to_numpy()          # pandas column -> NumPy array
growth = gen_old["growth_rate"].to_numpy()
dates = gen_old["date"].to_numpy()

print(f"{idx.size} months; mean index {idx.mean():.1f}; "
      f"highest {idx.max():.1f} in {pd.Timestamp(dates[idx.argmax()]):%B %Y}")

In [ ]:
falling = growth < 0                         # True where output fell y-o-y
print(falling[:6])
print(f"Months with falling output: {falling.sum()} of {falling.size} "
      f"({falling.mean():.0%})")

covid = (dates >= np.datetime64("2020-03-01")) & (dates <= np.datetime64("2021-03-01"))
print(f"...of which in March 2020 - March 2021: {(falling & covid).sum()}")
print(f"Months with no published growth rate: {np.isnan(growth).sum()}")
print(f"Average growth outside that period: {np.nanmean(growth[~covid]):.1f}%")

## 5.4 Laspeyres, Paasche and Fisher

In [ ]:
from indiadata import mospi_nas, NAS_REVISIONS


def nas_total(indicator):
    """Current and constant price totals by fiscal year, latest estimate."""
    d = mospi_nas(indicator)
    if "institutional_sector" in d.columns:   # GFCF also has rows by sector
        d = d[d["institutional_sector"].isna()
              | d["institutional_sector"].str.startswith("Total")]
    rank = d["revision"].map({r: i for i, r in enumerate(NAS_REVISIONS)})
    d = d.assign(rank=rank).sort_values("rank").groupby("year").tail(1)
    return d.sort_values("year").set_index("year")[["current_price",
                                                    "constant_price"]]


parts = {k: nas_total(k) for k in ["PFCE", "GFCE", "GFCF"]}
years = parts["PFCE"].index.to_numpy()
V = np.column_stack([parts[k]["current_price"] for k in parts]).astype(float)
Q = np.column_stack([parts[k]["constant_price"] for k in parts]).astype(float)
print("V and Q have shape", V.shape, "(years x components)")

In [ ]:
P = V / Q                         # implicit price of each component, 2011-12 = 1
p0, q0 = P[0], Q[0]

laspeyres = (P * q0).sum(axis=1) / (p0 * q0).sum() * 100
paasche = (P * Q).sum(axis=1) / (p0 * Q).sum(axis=1) * 100
fisher = np.sqrt(laspeyres * paasche)

idx_tab = pd.DataFrame({"Laspeyres": laspeyres, "Paasche": paasche,
                        "Fisher": fisher}, index=years).round(2)
idx_tab.iloc[[0, 3, 6, 9, 12, 14]]

In [ ]:
print("Paasche == sum(V)/sum(Q):",
      np.allclose(paasche, V.sum(axis=1) / Q.sum(axis=1) * 100))
gap = laspeyres - paasche
print(f"Laspeyres minus Paasche: smallest {gap.min():.2f}, "
      f"largest {gap.max():.2f} index points")
print("Prices in", years[-1], "(2011-12 = 100):",
      dict(zip(parts, np.round(P[-1] * 100, 1).tolist())))   # P[-1]: last row

## 5.5 Two measures of inflation: CPI and the GDP deflator

In [ ]:
gdp = nas_total("GDP")
deflator = gdp["current_price"].to_numpy() / gdp["constant_price"].to_numpy() * 100

month = cpi["date"].dt.month.to_numpy()
year = cpi["date"].dt.year.to_numpy()
fy_start = np.where(month >= 4, year, year - 1)
cpi_fy = pd.Series(cpi["index"].to_numpy()).groupby(fy_start).agg(["mean", "count"])
cpi_fy = cpi_fy[cpi_fy["count"] == 12]       # keep complete fiscal years only
cpi_fy.index = [f"{y}-{str(y + 1)[-2:]}" for y in cpi_fy.index]

both = pd.DataFrame({"CPI": cpi_fy["mean"],
                     "GDP deflator": pd.Series(deflator, index=gdp.index)}).dropna()
infl = (both.pct_change() * 100).dropna().round(2)
infl["CPI minus deflator"] = infl["CPI"] - infl["GDP deflator"]
print("Average", infl.index[0], "to", infl.index[-1], "\n" +
      infl[["CPI", "GDP deflator"]].mean().round(2).to_string())
infl

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(infl))
ax.bar(x - 0.2, infl["CPI"], width=0.4, color="#1f4e79", label="CPI (Combined)")
ax.bar(x + 0.2, infl["GDP deflator"], width=0.4, color="#c55a11",
       label="GDP deflator")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, infl.index, rotation=45)
ax.set_ylabel("% change over previous fiscal year")
ax.set_title("Two measures of Indian inflation, fiscal years", loc="left",
             fontsize=13)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.1, "Source: MoSPI, CPI (base 2012) averaged over April-March; "
         "National Accounts (base 2011-12), GDP at current / constant prices.",
         fontsize=8, color="grey")
plt.show()

## 5.6 The CPI as a weighted average of its groups

In [ ]:
from pathlib import Path

import requests
from pypdf import PdfReader

PR_URL = ("https://mospi.gov.in/sites/default/files/press_release/"
          "cpi_pr_12feb15t.pdf")
pdf_path = Path("data/raw/cpi_pr_12feb15t.pdf")
if not pdf_path.exists():
    reply = requests.get(PR_URL, timeout=90, headers={
        "User-Agent": "python-for-indian-economics (teaching use)"})
    reply.raise_for_status()
    pdf_path.write_bytes(reply.content)

annex1 = PdfReader(pdf_path).pages[1].extract_text().splitlines()
weights = {}
for line in annex1:
    parts_ = line.split()
    # group lines start with a one-digit code ("1   Food and beverages ...");
    # sub-group lines start with codes like "1.1.01" and are skipped
    if parts_ and parts_[0] in {"1", "2", "3", "4", "5", "6"} and len(parts_) > 10:
        name = " ".join(parts_[1:-9])
        weights[name] = float(parts_[-3])     # 7th of 9 numbers: combined weight
print("Sum of the six group weights:", round(sum(weights.values()), 2))
weights

In [ ]:
from indiadata import mospi_cpi_groups

grp = mospi_cpi_groups("combined").drop_duplicates(["date", "group", "subgroup"])
grp = grp[grp["subgroup"].str.endswith("-Overall")]
wide = grp.pivot(index="date", columns="group", values="index")

api_names = {n.lower(): n for n in wide.columns}
order = [api_names[k.lower()] for k in weights]   # match PDF names to API names
G = wide[order].to_numpy()                        # months x 6 groups
w = np.array(list(weights.values()))
print("G:", G.shape, "| w:", w.shape)

In [ ]:
rebuilt = G @ w / w.sum()
official = wide["General"].to_numpy()

ok = ~np.isnan(G).any(axis=1)          # months with all six group indices
print(f"Months: {len(ok)}; usable: {ok.sum()}")
print("Missing group values in:", list(wide.index[~ok].strftime("%Y-%m")))

err = rebuilt[ok] - official[ok]
print(f"Rebuilt minus official: mean {err.mean():.3f}, "
      f"largest absolute {np.abs(err).max():.3f} index points")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
palette = ["#c55a11", "#7f7f7f", "#9db7d5", "#1f4e79", "#e3a87a", "#3f7f3f"]
for j, name in enumerate(order):              # G[:, j]: all rows, column j
    ax.plot(wide.index, G[:, j], color=palette[j], linewidth=1.6,
            label=f"{name} ({w[j]:.2f}%)")
ax.plot(wide.index, official, color="black", linewidth=2.2, label="General")
ax.set_title("CPI (Combined) by group, base 2012 = 100; legend shows weights",
             loc="left", fontsize=13)
ax.legend(frameon=False, fontsize=8.5, ncol=2, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Source: MoSPI, CPI group indices via eSankhyiki API "
         "(the Housing gap, Dec 2018-Sep 2019, is in the API);\nweights from "
         "the CPI press release of 12 February 2015, Annexure I.",
         fontsize=8, color="grey")
plt.show()

## 5.7 Rebasing an index

In [ ]:
ref = (dates >= np.datetime64("2022-04-01")) & (dates <= np.datetime64("2023-03-01"))
print("Months in reference period:", ref.sum())
factor_rebase = idx[ref].mean() / 100
idx_rebased = idx / factor_rebase
print(f"Old IIP average in 2022-23: {idx[ref].mean():.2f}")
growth_before = idx[12:] / idx[:-12] - 1
growth_after = idx_rebased[12:] / idx_rebased[:-12] - 1
print("Growth rates unchanged by rebasing:", np.allclose(growth_before, growth_after))

## 5.8 Splicing the IIP across base years

In [ ]:
iip_new = mospi_iip(base_year="2022-23")
gen_new = iip_new[iip_new["type"] == "General"].set_index("date")["index"]
old = pd.Series(idx, index=pd.DatetimeIndex(dates))

overlap = pd.DataFrame({"old": old, "new": gen_new}).dropna()
print(f"Overlap: {overlap.index[0]:%B %Y} to {overlap.index[-1]:%B %Y}, "
      f"{len(overlap)} months")
overlap.head(3)

In [ ]:
ratio = (overlap["old"] / overlap["new"]).to_numpy()
print(f"Ratio old/new: min {ratio.min():.3f} ({overlap.index[ratio.argmin()]:%b %Y}), "
      f"max {ratio.max():.3f} ({overlap.index[ratio.argmax()]:%b %Y})")

fy24 = (overlap.index >= "2023-04-01") & (overlap.index <= "2024-03-01")
candidates = {
    "April 2023 only (first month)": ratio[0],
    "March 2026 only (last month)": ratio[-1],
    "2023-24 average of indices": overlap["old"][fy24].mean() / overlap["new"][fy24].mean(),
    "2023-24 geometric mean of ratios": np.exp(np.log(ratio[fy24]).mean()),
    "whole overlap, average of ratios": ratio.mean(),
    "rebasing factor (old 2022-23 average / 100)": factor_rebase,
}
factors = pd.Series(candidates).round(4)
factors

In [ ]:
last = PdfReader(pdf_path).pages[6].extract_text()
print([ln for ln in last.splitlines() if ln.strip().startswith("All Groups")])

In [ ]:
f_year = factors["2023-24 average of indices"]
f_first = factors["April 2023 only (first month)"]
f_last = factors["March 2026 only (last month)"]


def splice(factor):
    back = old[old.index < gen_new.index[0]] / factor      # old, rescaled
    return pd.concat([back, gen_new])                       # then new


spliced = splice(f_year)
print(f"Spliced series: {spliced.index[0]:%b %Y} to {spliced.index[-1]:%b %Y}, "
      f"{len(spliced)} months")
for label, f in [("first month", f_first), ("2023-24 year", f_year),
                 ("last month", f_last)]:
    print(f"April 2012 on new base, linking on {label:12s}: "
          f"{splice(f).iloc[0]:.1f}")

In [ ]:
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [2.2, 1]})
ax1.plot(old.index, old, color="#9db7d5", linewidth=1.2,
         label="Old base (2011-12 = 100)")
ax1.plot(gen_new.index, gen_new, color="#c55a11", linewidth=1.6,
         label="New base (2022-23 = 100)")
ax1.plot(spliced.index, spliced, color="#1f4e79", linewidth=1.2,
         linestyle="--", label="Spliced, linked on 2023-24")
ax1.set_title("IIP general index: two bases, one spliced series", loc="left",
              fontsize=12)
ax1.legend(frameon=False, fontsize=8.5, loc="upper left")
ax1.spines[["top", "right"]].set_visible(False)

ax2.plot(overlap.index, ratio, color="#1f4e79", marker="o", markersize=3)
ax2.axhline(f_year, color="#c55a11", linestyle="--", linewidth=1,
            label=f"2023-24 factor ({f_year:.3f})")
ax2.legend(frameon=False, fontsize=8.5, loc="lower left")
ax2.set_title("Splice factor by linking month", loc="left", fontsize=12)
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[4, 10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
ax2.tick_params(axis="x", labelrotation=45)
ax2.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Source: MoSPI, Index of Industrial Production, monthly, "
         "bases 2011-12 and 2022-23, via eSankhyiki API.", fontsize=8,
         color="grey")
fig.tight_layout()
plt.show()

In [ ]:
def fiscal_year(dt_index):
    """First calendar year of the April-March fiscal year of each date."""
    return np.where(dt_index.month >= 4, dt_index.year, dt_index.year - 1)


fy_avg = pd.DataFrame({
    "old base": old.groupby(fiscal_year(old.index)).mean(),
    "new base": gen_new.groupby(fiscal_year(gen_new.index)).mean(),
}).loc[2023:2025]
fy_avg.index = ["2023-24", "2024-25", "2025-26"]
(fy_avg.pct_change() * 100).round(2).dropna()

## Key ideas

## Exercises

## Data citation

## Further reading